# 20 — Gold Target Engineering

**Objetivo:** congelar y materializar el target V1 de ARGOS con semántica explícita, horizonte temporal exacto y censura correcta.

Este notebook consume exclusivamente la capa Silver cerrada en M2. No crea features ni splits.

## Semantic freeze V1

`y_addition_180d = 1` significa que existe al menos un evento canónico `ADICION EN EL VALOR` en la ventana **(t0, t0 + 180 días]**, donde `t0 = fecha_de_firma`.

Los contratos sin seguimiento completo de 180 días son **censurados** y no se convierten en negativos.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.target import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

In [ ]:
from argos.gold.target import build_target_gold

result = build_target_gold(
    silver_dir=paths.silver,
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}

## Auditoría del target

In [ ]:
display(result.event_scope)
display(result.censoring)
display(result.by_year)

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("20 ✅ PASS" if result.ready_for_next_notebook else "20 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_next_notebook, "Target Gold no supera sus checks bloqueantes."

## Criterio de salida

No avanzar a `21_temporal_split_design.ipynb` si `20` no termina en `✅ PASS`. El target queda congelado documentalmente en `docs/target-specs/y_addition_180d.md`.